# Fase 4. Regresión lineal de MolLogP

En esta fase tomo MolLogP como variable dependiente y evalúo si otros descriptores moleculares ayudan a explicarlo. Considero todos los descriptores numéricos de RDKit disponibles, retiro la respuesta, variables de identificación y LipinskiViolations, y elimino constantes y predictores redundantes usando solo entrenamiento. Reservo prueba para la evaluación final.

## 4.1. Lectura de los datos

Cargo los CSV preparados en la fase 2 desde el repositorio.

In [ ]:
# Cargo las librerías para leer y analizar las tablas.
from pathlib import Path  # Organizo las rutas locales.
import pandas as pd  # Leo y preparo tablas.
import numpy as np  # Calculo estadísticas numéricas.
from IPython.display import display  # Muestro resultados en Colab.

# Defino una carpeta local para guardar copias de trabajo.
DATA_DIR = Path('/content/datos_dili')  # Uso la carpeta de Colab.
DATA_DIR.mkdir(parents=True, exist_ok=True)  # Creo la carpeta si hace falta.

# Indico la carpeta pública con los datos que preparé en la fase 2.
BASE_URL = 'https://raw.githubusercontent.com/PARODBE/UAX_Fundamento_Ciencia_de_Datos/main/Data/'  # Defino el enlace base.
dilist_original = pd.read_csv(BASE_URL + 'DILIst_FDA.csv')  # Cargo la fuente original.
estructuras = pd.read_csv(BASE_URL + 'estructuras_nombres.csv')  # Cargo el mapa de nombres y estructuras.
datos_limpios = pd.read_csv(BASE_URL + 'dilist_descriptores.csv')  # Cargo los compuestos con todos los descriptores.
train_df = pd.read_csv(BASE_URL + 'dilist_train.csv')  # Cargo entrenamiento.
test_df = pd.read_csv(BASE_URL + 'dilist_test.csv')  # Cargo prueba.

# Creo una tabla completa para revisar cobertura y calidad de estructuras.
dilist = dilist_original.rename(columns={'DILIst_Classification': 'DILI'}).copy()  # Renombro la clase.
datos = dilist.merge(estructuras, on='CompoundName', how='left')  # Uno las etiquetas con el estado de búsqueda.
claves_con_etiqueta = datos.dropna(subset=['InChIKey']).groupby('InChIKey')['DILI'].nunique()  # Cuento clases por estructura.
claves_conflictivas = claves_con_etiqueta[claves_con_etiqueta > 1].index  # Identifico etiquetas contradictorias.
datos_conflicto = datos[datos['InChIKey'].isin(claves_conflictivas)].copy()  # Guardo los registros conflictivos.

# Identifico todos los descriptores numéricos salvo etiquetas e identificadores.
columnas_excluir = ['DILI', 'DILIST_ID', 'InChIKey', 'CID', 'SMILES', 'CompoundName', 'Routes_of_Administration', 'chembl_id']  # Evito datos no moleculares.
DESCRIPTORES = [columna for columna in datos_limpios.select_dtypes(include='number').columns if columna not in columnas_excluir]  # Selecciono propiedades moleculares.

# Guardo copias locales de las tablas de trabajo.
for nombre, tabla in [('dilist_descriptores.csv', datos_limpios), ('dilist_train.csv', train_df), ('dilist_test.csv', test_df)]:  # Recorro las tablas.
    tabla.to_csv(DATA_DIR / nombre, index=False)  # Guardo una copia en Colab.
print('He cargado los datos del repositorio:', DATA_DIR)  # Confirmo la ruta local.
print('Descriptores numéricos disponibles:', len(DESCRIPTORES))  # Muestro el catálogo disponible.

## 4.2. Preparación y ajuste

Primero reviso la correlación entre predictores con entrenamiento y retiro variables redundantes con correlación absoluta de Spearman de 0,90 o más. Ajusto la imputación y la estandarización solo con train; después aplico esas transformaciones a train y test.

In [ ]:
# Importo herramientas para imputar, estandarizar y ajustar una regresión lineal.
from sklearn.impute import SimpleImputer  # Completo posibles valores ausentes.
from sklearn.preprocessing import StandardScaler  # Estandarizo los predictores.
from statsmodels.api import OLS, add_constant  # Ajusto la regresión y añado el intercepto.

# Defino MolLogP como variable que quiero explicar.
respuesta = 'MolLogP'  # Selecciono la variable dependiente.
columnas_excluir = ['MolLogP', 'LipinskiViolations', 'DILI', 'DILIST_ID']  # Evito respuesta, su derivación y variables de identificación.
predictores = [variable for variable in DESCRIPTORES if variable not in columnas_excluir and train_df[variable].nunique(dropna=True) > 1]  # Uso descriptores no constantes.
print('Descriptores candidatos:', len(predictores))  # Informo el tamaño inicial del catálogo.

# Retiro predictores redundantes con correlación alta en entrenamiento.
correlacion = train_df[predictores].corr(method='spearman').abs()  # Calculo asociaciones absolutas en train.
seleccionados = []  # Creo una lista para los predictores finales.
for variable in predictores:  # Reviso los predictores disponibles.
    if not any(correlacion.loc[variable, otra] >= 0.90 for otra in seleccionados):  # Compruebo redundancia con los ya elegidos.
        seleccionados.append(variable)  # Conservo variables que no superan el límite.
print('Descriptores conservados:', len(seleccionados))  # Muestro cuántas variables entran al modelo.

# Preparo respuesta e imputación usando solo entrenamiento.
y_train = train_df[respuesta]  # Tomo MolLogP de entrenamiento.
y_test = test_df[respuesta]  # Tomo MolLogP de prueba.
imputador = SimpleImputer(strategy='median')  # Creo imputación por mediana.
imputador.fit(train_df[seleccionados])  # Aprendo medianas solo con train.
X_train_imp = imputador.transform(train_df[seleccionados])  # Transformo los predictores de train.
X_test_imp = imputador.transform(test_df[seleccionados])  # Aplico medianas de train a test.
escalador = StandardScaler()  # Creo el estandarizador.
escalador.fit(X_train_imp)  # Aprendo medias y desviaciones solo con train.
X_train = pd.DataFrame(escalador.transform(X_train_imp), columns=seleccionados, index=train_df.index)  # Estandarizo train.
X_test = pd.DataFrame(escalador.transform(X_test_imp), columns=seleccionados, index=test_df.index)  # Transformo test con el mismo escalador.

# Ajusto el modelo lineal con entrenamiento.
modelo = OLS(y_train, add_constant(X_train, has_constant='add')).fit()  # Estimo coeficientes e inferencia.
print(modelo.summary())  # Muestro intercepto, coeficientes, R cuadrado, F y valores p.

## 4.3. Evaluación en prueba

Calculo R² y RMSE en los datos reservados. Los coeficientes representan el cambio esperado de MolLogP por una desviación estándar del predictor, manteniendo constantes los demás.

In [ ]:
# Importo métricas para medir el error en el conjunto de prueba.
from sklearn.metrics import mean_squared_error, r2_score  # Calculo error cuadrático y R cuadrado.

# Predigo MolLogP para compuestos que no se usaron en el ajuste.
prediccion = modelo.predict(add_constant(X_test, has_constant='add'))  # Calculo valores predichos.
mse = mean_squared_error(y_test, prediccion)  # Calculo el error cuadrático medio.
rmse = np.sqrt(mse)  # Expreso el error en unidades de MolLogP.
r2_test = r2_score(y_test, prediccion)  # Calculo la proporción de varianza explicada en test.
print(f'R² de prueba: {r2_test:.3f}')  # Informo R cuadrado.
print(f'MSE de prueba: {mse:.3f}')  # Informo error cuadrático medio.
print(f'RMSE de prueba: {rmse:.3f}')  # Informo error típico aproximado.

# Comparo valores observados y predichos en una figura.
import matplotlib.pyplot as plt  # Importo gráficos.
plt.scatter(y_test, prediccion, alpha=0.65)  # Dibujo observaciones y predicciones.
plt.xlabel('MolLogP observado')  # Etiqueto el eje horizontal.
plt.ylabel('MolLogP predicho')  # Etiqueto el eje vertical.
plt.title('Regresión lineal: valores observados y predichos')  # Titulo el gráfico.
plt.tight_layout()  # Ajusto los márgenes.
plt.show()  # Muestro la figura.


## Lectura del modelo

Interpreto la magnitud y el signo de los coeficientes junto con sus valores p y el estadístico F. Considero R² y RMSE de prueba para valorar la capacidad predictiva fuera de la muestra. Como MolLogP es una propiedad fisicoquímica, el análisis no representa una predicción clínica de lesión hepática.